# Amazon Bedrock AgentCore Gateway와 Amazon Bedrock AgentCore Runtime 통합

## 개요
[Amazon Bedrock AgentCore Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html)는 기존 AWS Lambda 함수와 API(OpenAPI 및 Smithy)를 인프라나 호스팅 관리 없이 완전 관리형 MCP 서버로 전환할 수 있는 방법을 제공합니다. [Amazon Bedrock AgentCore Runtime](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agents-tools-runtime.html)은 AI 에이전트 또는 도구를 배포하고 실행할 수 있도록 설계된 안전한 서버리스 호스팅 환경을 제공합니다. 이 튜토리얼에서는 Amazon Bedrock AgentCore Gateway를 AgentCore Runtime 및 [Strands Agents](https://strandsagents.com/latest/)와 통합합니다. 

### 튜토리얼 세부 정보


| 정보                 | 세부 정보                                                 |
|:---------------------|:----------------------------------------------------------|
| 튜토리얼 유형        | 대화형                                                    |
| AgentCore 구성 요소  | AgentCore Gateway, AgentCore Identity, AgentCore Runtime  |
| 에이전트 프레임워크  | Strands Agents                                            |
| Gateway 대상 유형    | AWS Lambda, OpenAPI 대상                                  |
| 인바운드 인증 IdP    | AWS IAM                                                   |
| 아웃바운드 인증      | AWS IAM (AWS Lambda), API Key (OpenAPI 대상)              |
| LLM 모델             | Anthropic Claude Haiku 4.5, Amazon Nova Pro              |
| 튜토리얼 구성 요소   | AgentCore Gateway 생성 및 호출                            |
| 튜토리얼 분야        | 여러 분야 공통                                            |
| 예제 난이도          | 중간                                                       |
| 사용한 SDK           | boto3                                                     |


### 튜토리얼 아키텍처
이 튜토리얼에서는 AWS Lambda 함수와 RESTful API에 정의된 작업을 MCP 도구로 변환하고 Amazon Bedrock AgentCore Gateway에서 호스팅합니다. AWS SigV4 형식의 AWS IAM 자격 증명을 사용하는 인바운드 인증을 살펴봅니다. 또한 AgentCore Gateway 도구를 활용하는 Strands Agent를 AgentCore Runtime에 배포합니다. 

실습에서는 [Amazon Bedrock](https://aws.amazon.com/bedrock/) 모델을 사용하는 Strands Agent를 활용합니다. 

<center>

![Runtime과 Gateway](./images/runtime_gateway.png)

</center>

## 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Python 3.10+
* AWS 자격 증명
* Amazon Bedrock AgentCore SDK
* Strands Agents

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

: 

In [ ]:
import os
import sys

if "__file__" in globals():
    current_dir = os.path.dirname(os.path.abspath(__file__))
else:
    current_dir = os.getcwd()

# 공유 utils 모듈에 접근할 수 있도록 두 단계 상위 디렉터리로 이동합니다.
# utils.py 파일에는 여러 튜토리얼에서 사용하는 도우미 함수가 포함되어 있습니다.
utils_dir = os.path.abspath(os.path.join(current_dir, "../.."))

# utils 디렉터리를 Python 모듈 검색 경로에 추가합니다.
# 이렇게 하면 상위 디렉터리에서 utils를 가져올 수 있습니다.
sys.path.insert(0, utils_dir)

In [ ]:
# Gateway 관리용 도우미 함수가 포함된 utils 모듈을 가져옵니다.
import utils

In [ ]:
# 튜토리얼에 필요한 모든 라이브러리를 가져옵니다.
from bedrock_agentcore_starter_toolkit import Runtime
from bedrock_agentcore_starter_toolkit.operations.runtime import (
    destroy_bedrock_agentcore,
)
from deploy_cloudformation import deploy_stack, delete_stack
from pathlib import Path
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp.mcp_client import MCPClient
from streamable_http_sigv4 import streamablehttp_client_with_sigv4
import boto3
import getpass
import json
import logging
import uuid

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

In [ ]:
session = boto3.Session()

credentials = session.get_credentials()

region = session.region_name

cf_client = boto3.client("cloudformation", region_name=region)

agentcore_client = boto3.client(
    "bedrock-agentcore-control",
    region_name=region,
)

identity_client = boto3.client(
    "bedrock-agentcore-control",
    region_name=region,
)

s3_client = session.client("s3")

sts_client = session.client("sts")
account_id = sts_client.get_caller_identity()["Account"]

In [ ]:
# 튜토리얼 리소스의 구성 변수를 정의합니다.

# CloudFormation 스택 구성
stack_name = "customer-support-lambda-stack"  # Lambda와 DynamoDB를 생성하는 스택의 이름
template_file = "cloudformation/customer_support_lambda.yaml"  # CloudFormation 템플릿 경로

# Gateway 및 대상 이름
gateway_name = "customer-support-gateway"  # AgentCore Gateway 이름
open_api_target_name = "DemoOpenAPITargetS3NasaMars"  # NASA API Gateway 대상 이름
lambda_target_name = "LambdaUsingSDK"  # Lambda 함수 Gateway 대상 이름

# OpenAPI 사양을 저장할 S3 버킷 구성
unique_s3_name = str(uuid.uuid4())  # 버킷에 사용할 전역 고유 식별자를 생성합니다.
bucket_name = f"agentcore-gateway-{unique_s3_name}"  # 명확히 구분할 수 있도록 'agentcore-gateway' 접두사를 사용합니다.
file_path = "openapi-specs/nasa_mars_insights_openapi.json"  # OpenAPI 사양의 로컬 경로
object_key = "nasa_mars_insights_openapi.json"  # S3 객체 키(버킷 내 파일 이름)

# NASA API 인증용 자격 증명 공급자 이름
api_key_credential_provider_name = "NasaInsightAPIKey"  # NASA API 키를 안전하게 저장합니다.

# 에이전트 구성
agent_name = "customer_support_gateway"  # AgentCore Runtime에 배포되는 에이전트 이름

# 필요한 경우 다른 Notebook에서 사용할 수 있도록 고유한 S3 이름을 저장합니다.
%store unique_s3_name

## 1단계: AWS Lambda 및 Amazon DynamoDB 배포

### AWS CloudFormation 스택 리소스

[AWS CloudFormation](https://docs.aws.amazon.com/AWSCloudFormation/latest/UserGuide/Welcome.html) 템플릿은 다음 AWS 리소스를 배포합니다.

1. **AgentCoreRuntimeExecutionRole**: 다음 권한을 포함하는 AgentCore Runtime 실행 역할
   - 컨테이너 배포를 위한 ECR 이미지 접근
   - 모니터링 및 디버깅을 위한 CloudWatch Logs
   - 관찰성을 위한 X-Ray 추적
   - AI 기능을 위한 Bedrock 모델 호출
   - MCP 도구 호출을 위한 Gateway 호출
   - 워크로드 자격 증명 토큰 생성

2. **GatewayAgentCoreRole**: 다음 권한을 포함하는 AgentCore Gateway 실행 역할
   - Lambda 함수 호출
   - OpenAPI 사양을 위한 S3 접근
   - 자격 증명 검색을 위한 Secrets Manager 접근
   - 모든 AgentCore 및 Bedrock 작업
   - confused deputy 방지를 위한 강화된 신뢰 정책 포함

3. **CustomerSupportLambda**: 다음 기능을 제공하는 기본 Lambda 함수
   - `get_customer_profile`: ID, 이메일 또는 전화번호로 고객 정보 조회
   - `check_warranty_status`: 일련번호로 제품 보증 확인
   
4. **PopulateDataFunction**: DynamoDB 테이블에 초기 데이터를 입력하는 사용자 지정 리소스 Lambda

5. **CustomerProfileTable**: 다음 구조로 고객 정보를 저장
   - 기본 키: `customer_id`
   - 유연한 조회를 위한 `email` 및 `phone`의 Global Secondary Index
   - 고객 프로필 샘플 5개 포함

6. **WarrantyTable**: 다음 구조로 제품 보증 정보를 저장
   - 기본 키: `serial_number`
   - 고객별 쿼리를 위한 `customer_id`의 Global Secondary Index
   - 보증 레코드 샘플 8개 포함

이 스택은 튜토리얼에서 사용할 ARN 세 개를 내보냅니다.
- `CustomerSupportLambdaArn`: Lambda 함수의 ARN(Gateway 대상)
- `GatewayAgentCoreRoleArn`: Gateway 실행용 ARN
- `AgentCoreRuntimeExecutionRoleArn`: Runtime 에이전트 배포용 ARN

모든 리소스에는 비용 추적과 관리를 위한 적절한 태그가 포함됩니다.

In [ ]:
# CloudFormation 스택을 배포합니다.
lambda_arn, gateway_role_arn, runtime_execution_role_arn = deploy_stack(
    stack_name=stack_name,
    template_file=template_file,
    region=region,
    cf_client=cf_client,
)

## 2단계: 인바운드 요청의 권한 부여 방식으로 IAM을 사용하는 AgentCore Gateway 생성

AgentCore Gateway는 인바운드 인증에 [AWS IAM](https://aws.amazon.com/iam/) 지원을 도입하여 기존의 OAuth 전용 방식에서 인증 기능을 크게 확장했습니다. [MCP 프로토콜](https://modelcontextprotocol.io/docs/getting-started/intro) 사양은 전통적으로 인증에 OAuth 토큰을 요구하지만, 이 새로운 기능을 사용하면 MCP 서버로 들어오는 요청에 AWS IAM 자격 증명을 사용할 수 있어 중요한 엔터프라이즈 요구 사항을 충족할 수 있습니다.

이전에는 Gateway 개발자가 권한 부여 유형으로 CUSTOM_JWT만 구성할 수 있었기 때문에 모든 인바운드 요청에 OAuth 토큰 기반 인증이 필요했습니다. 이제 개발자는 권한 부여 유형으로 AWS_IAM을 선택하여 인바운드 MCP 요청에 AWS Signature Version 4(SigV4) 인증을 사용할 수 있습니다. 

이 구현에는 두 가지 주요 IAM 구성 요소가 도입됩니다.

- Gateway 요청 권한 부여를 위한 새 작업 `bedrock-agentcore:InvokeGateway`
- 권한 부여 유형에 따라 Gateway 생성을 제어하는 새 조건 키 `bedrock-agentcore:GatewayAuthorizerType`

In [ ]:
# AWS IAM을 권한 부여 방식으로 사용하는 AgentCore Gateway를 생성합니다.
# 핵심 기능은 인증에 CUSTOM_JWT 대신 AWS_IAM을 사용하는 것입니다.
create_response = agentcore_client.create_gateway(
    name=gateway_name,
    roleArn=gateway_role_arn,
    protocolType="MCP",
    authorizerType="AWS_IAM",
    description="AgentCore Gateway with AWS Lambda target type using AWS IAM for ingress auth",
)
logger.info(f"Gateway created: {create_response}")

gateway_id = create_response["gatewayId"]
gateway_url = create_response["gatewayUrl"]
logger.info(f"Gateway ID: {gateway_id}")
logger.info(f"Gateway URL: {gateway_url}")

## 3단계: Amazon Bedrock AgentCore Gateway를 사용해 고객 지원 AWS Lambda를 MCP 도구로 변환

<center>

![Lambda 및 DynamoDB 구성](./images/lambda_dynamodb.png)

</center>

In [ ]:
# Lambda 함수를 Gateway 대상으로 구성합니다.
# 이렇게 하면 Lambda 함수 작업이 AI 에이전트가 사용할 수 있는 MCP 도구로 변환됩니다.
lambda_target_config = {
    "mcp": {
        "lambda": {
            "lambdaArn": lambda_arn,
            "toolSchema": {
                "inlinePayload": [
                    {
                        # 첫 번째 도구: 고객 프로필 정보 조회
                        "name": "get_customer_profile",
                        "description": "Retrieve customer profile using customer ID, email, or phone number",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                # ID, 이메일 또는 전화번호로 고객을 조회할 수 있습니다.
                                "customer_id": {"type": "string"},
                                "email": {"type": "string"},
                                "phone": {"type": "string"},
                            },
                            # 최소한 customer_id가 필요합니다.
                            "required": ["customer_id"],
                        },
                    },
                    {
                        # 두 번째 도구: 제품 보증 상태 확인
                        "name": "check_warranty_status",
                        "description": "Check the warranty status of a product using its serial number and optionally verify via email",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                # 일련번호로 제품을 고유하게 식별합니다.
                                "serial_number": {"type": "string"},
                                # 고객 이메일을 확인에 사용할 수 있습니다.
                                "customer_email": {"type": "string"},
                            },
                            # 일련번호는 필수입니다.
                            "required": ["serial_number"],
                        },
                    },
                ]
            },
        }
    }
}

# Gateway가 Lambda 함수에 인증하는 방식을 구성합니다.
# Gateway의 IAM 역할(gateway_role_arn)을 사용하여 Lambda를 호출합니다.
credential_config = [{"credentialProviderType": "GATEWAY_IAM_ROLE"}]

# Gateway 대상을 생성하여 Lambda 함수를 MCP 도구로 사용할 수 있게 합니다.
response = agentcore_client.create_gateway_target(
    gatewayIdentifier=gateway_id,
    name=lambda_target_name,
    description="Lambda Target using SDK",
    targetConfiguration=lambda_target_config,
    credentialProviderConfigurations=credential_config,
)

## 4단계: Amazon Bedrock AgentCore Gateway를 사용해 NASA 공개 API를 MCP 도구로 변환


<center>

![NASA API 구성](./images/nasa.png)

</center>

NASA 공개 API에서 날씨 데이터를 가져오는 화성 날씨 에이전트를 만들어 보겠습니다. NASA InSight API를 사용하려면 [여기](https://api.nasa.gov/)에서 무료로 등록해야 합니다. 등록하면 이메일로 API Key를 받게 됩니다. 이 API Key를 사용하여 OpenAPI 대상을 생성할 자격 증명 공급자를 구성합니다.



### 4.1단계: API 자격 증명 공급자 생성 - Amazon Bedrock AgentCore Identity

In [ ]:
# 사용자에게 NASA API Key를 안전하게 입력하도록 요청합니다.
# getpass는 입력값이 화면에 표시되지 않도록 숨깁니다.
nasa_api_key = getpass.getpass(prompt="Enter your NASA API Key: ")

In [ ]:
if not nasa_api_key:
    logger.error("NASA API Key is required. Please run the cell above and enter your API key.")
    raise ValueError("NASA API Key is required")

# API 키를 안전하게 저장할 자격 증명 공급자를 AgentCore Identity에 생성합니다.
# 이를 통해 Gateway는 키를 노출하지 않고 NASA API에 인증할 수 있습니다.
response = identity_client.create_api_key_credential_provider(
    name=api_key_credential_provider_name,
    apiKey=nasa_api_key,
)

logger.info(f"Credential provider response: {response}")

credential_provider_arn = response["credentialProviderArn"]
logger.info(f"Egress Credentials provider ARN: {credential_provider_arn}")

### 4.2단계: [NASA OpenAPI 사양](./openapi-specs/nasa_mars_insights_openapi.json)을 업로드할 Amazon S3 버킷 생성

In [ ]:
try:
    # OpenAPI 사양 파일을 저장할 S3 버킷을 생성합니다.
    if region == "us-east-1":
        # us-east-1에는 LocationConstraint가 필요하지 않습니다.
        s3bucket = s3_client.create_bucket(Bucket=bucket_name)
    else:
        # 그 외 모든 리전은 리전을 지정하기 위해 LocationConstraint가 필요합니다.
        s3bucket = s3_client.create_bucket(Bucket=bucket_name, CreateBucketConfiguration={"LocationConstraint": region})

    # OpenAPI 사양 JSON 파일을 S3에 업로드합니다.
    # Gateway는 이 파일을 읽어 NASA API 구조를 파악합니다.
    with open(file_path, "rb") as file_data:
        response = s3_client.put_object(Bucket=bucket_name, Key=object_key, Body=file_data)

    openapi_s3_uri = f"s3://{bucket_name}/{object_key}"
    print(f"Uploaded object S3 URI: {openapi_s3_uri}")
except Exception as e:
    print(f"Error uploading file: {e}")

### 4.3단계: 아웃바운드 인증 구성 및 Gateway 대상 생성


In [ ]:
# NASA OpenAPI 대상을 구성합니다.
# 이렇게 하면 NASA의 Mars InSight API가 MCP 도구로 변환됩니다.
nasa_openapi_s3_target_config = {"mcp": {"openApiSchema": {"s3": {"uri": openapi_s3_uri}}}}

# NASA로 전송되는 아웃바운드 요청에 대한 API 키 인증을 구성합니다.
# Gateway는 NASA API로 보내는 모든 요청에 API 키를 추가합니다.
api_key_credential_config = [
    {
        "credentialProviderType": "API_KEY",
        "credentialProvider": {
            "apiKeyCredentialProvider": {
                # NASA는 API 키가 "api_key"라는 쿼리 파라미터로 전달되기를 요구합니다.
                "credentialParameterName": "api_key",
                # 앞에서 생성한 자격 증명 공급자의 ARN
                "providerArn": credential_provider_arn,
                # NASA API는 키가 헤더가 아닌 쿼리 문자열에 포함되기를 요구합니다.
                "credentialLocation": "QUERY_PARAMETER",  # 옵션: "HEADER" 또는 "QUERY_PARAMETER"
                # 참고: credentialPrefix("Basic" 또는 "Bearer" 등)는 헤더 기반 인증에 사용됩니다.
                # "credentialPrefix": " "  # header 기반 token 인증을 사용하면 주석을 해제합니다.
            }
        },
    }
]

# OpenAPI Gateway 대상을 생성합니다.
# 이렇게 하면 모든 NASA API 작업을 MCP 도구로 사용할 수 있습니다.
response = agentcore_client.create_gateway_target(
    gatewayIdentifier=gateway_id,
    name=open_api_target_name,
    description="OpenAPI Target with S3Uri using SDK",
    targetConfiguration=nasa_openapi_s3_target_config,
    credentialProviderConfigurations=api_key_credential_config,
)

## 5단계: Strands Agent에서 MCP 도구 호출

#### MCP Client SDK의 AWS IAM 인증 지원
MCP Client SDK의 AWS IAM 인증 지원

이제 AgentCore Gateway의 인바운드 요청에 AWS IAM 인증을 사용할 수 있지만, 현재 오픈 소스 MCP Client SDK의 SigV4 인증 지원은 특히 스트리밍 가능한 HTTP 연결에서 제한적입니다. AWS는 스트리밍 HTTP 연결의 SigV4 인증에 필요한 핵심 확장이 포함된 "Run Model Context Protocol (MCP) servers with AWS Lambda" 프로젝트를 통해 해결 방법을 제공합니다.

[AWS Labs GitHub 리포지토리](https://github.com/awslabs/run-model-context-protocol-servers-with-aws-lambda/tree/main)에서 제공하는 이 구현은 스트리밍 연결의 인증 공백을 해소하며 Strands 또는 LangChain과 같은 주요 에이전트 프레임워크에 원활하게 통합할 수 있습니다. StreamableHTTPTransportWithSigV4 클래스는 표준 MCP 전송 계층을 확장하여 스트리밍 기능을 유지하면서 AWS SigV4 서명을 처리하므로 AgentCore Gateway의 새로운 IAM 인증 기능과 호환됩니다.

In [ ]:
def create_streamable_http_transport_sigv4(mcp_url: str, service_name: str, region: str):
    """
    AWS SigV4 인증을 사용하는 streamable HTTP transport를 생성합니다.

    이 함수는 AWS Signature Version 4(SigV4)로 요청을 인증하는 MCP client transport를
    생성합니다. 표준 MCP 클라이언트는 AWS IAM 인증을 기본 지원하지 않으므로,
    이 transport가 그 간극을 연결합니다.

    매개변수:
        mcp_url (str): MCP gateway endpoint URL
        service_name (str): SigV4 서명에 사용할 AWS 서비스 이름(일반적으로 "bedrock-agentcore")
        region (str): gateway가 배포된 AWS 리전

    반환값:
        StreamableHTTPTransportWithSigV4: SigV4 인증용으로 구성된 transport 인스턴스

    예시:
        >>> transport = create_streamable_http_transport_sigv4(
        ...     mcp_url="https://gateway-id.gateway.region.aws.dev/mcp",
        ...     service_name="bedrock-agentcore",
        ...     region="us-west-2"
        ... )
    """
    # 현재 boto3 세션에서 AWS 자격 증명을 가져옵니다.
    # 이 자격 증명은 SigV4로 요청에 서명하는 데 사용됩니다.
    session = boto3.Session()
    credentials = session.get_credentials()

    # SigV4 서명 기능을 갖춘 사용자 지정 전송을 생성하여 반환합니다.
    return streamablehttp_client_with_sigv4(
        url=mcp_url,
        credentials=credentials,
        service=service_name,
        region=region,
    )


def get_full_tools_list(client):
    """
    페이지네이션을 처리해 MCP 클라이언트의 전체 도구 목록을 조회합니다.

    MCP 서버는 도구를 페이지 단위 응답으로 반환할 수 있습니다. 이 함수는 페이지네이션을
    자동으로 처리하고 사용 가능한 모든 도구를 하나의 목록으로 반환합니다.

    매개변수:
        client: MCP 클라이언트 인스턴스(strands.tools.mcp.mcp_client.MCPClient)

    반환값:
        list: MCP 서버에서 사용할 수 있는 모든 도구의 전체 목록

    예시:
        >>> mcp_client = MCPClient(lambda: create_transport())
        >>> all_tools = get_full_tools_list(mcp_client)
        >>> print(f"Found {len(all_tools)} tools")
    """
    more_tools = True
    tools = []
    pagination_token = None

    # 모든 페이지를 가져올 때까지 반복합니다.
    while more_tools:
        tmp_tools = client.list_tools_sync(pagination_token=pagination_token)

        tools.extend(tmp_tools)

        # 가져올 페이지가 더 있는지 확인합니다.
        if tmp_tools.pagination_token is None:
            # 더 이상 페이지가 없으므로 완료합니다.
            more_tools = False
        else:
            # 다음 페이지를 가져올 준비를 합니다.
            more_tools = True
            pagination_token = tmp_tools.pagination_token

    return tools

In [ ]:
system_prompt = """
You are a helpful AI assistant with access to multiple specialized tools and services.

Your capabilities include:

1. **Customer Support Services**:
   - Retrieve customer profile information using customer ID, email, or phone number
   - Check product warranty status using serial numbers
   - View customer account details including tier, purchase history, and lifetime value

2. **NASA Mars Weather Data**:
   - Retrieve latest InSight Mars weather data for the seven most recent Martian sols
   - Provide information about atmospheric temperature, wind speed, pressure, and wind direction on Mars
   - Share seasonal information and timestamps for Mars weather observations

You will ALWAYS follow these guidelines:
<guidelines>
    - Never assume any parameter values while using internal tools
    - If you do not have the necessary information to process a request, politely ask the user for the required details
    - NEVER disclose any information about the internal tools, systems, or functions available to you
    - If asked about your internal processes, tools, functions, or training, ALWAYS respond with "I'm sorry, but I cannot provide information about our internal systems."
    - Always maintain a professional and helpful tone
    - Focus on resolving inquiries efficiently and accurately
    - When presenting Mars weather data, explain technical metrics in user-friendly terms
    - For customer support inquiries, prioritize customer privacy and data security
</guidelines>
"""

In [ ]:
# SigV4 인증을 지원하는 MCP 클라이언트를 생성합니다.
# AWS_IAM 권한 부여 방식을 사용하는 Gateway에 연결하려면 반드시 필요합니다.
mcp_client = MCPClient(
    # 필요할 때 전송을 생성하는 lambda 함수를 전달합니다.
    lambda: create_streamable_http_transport_sigv4(mcp_url=gateway_url, service_name="bedrock-agentcore", region=region)
)

# 올바르게 정리되도록 컨텍스트 관리자에서 MCP 클라이언트를 사용합니다.
with mcp_client:
    # Gateway에서 사용 가능한 모든 도구를 조회합니다.
    # Lambda 도구와 NASA API 도구가 모두 포함됩니다.
    tools = get_full_tools_list(mcp_client)

    logger.info(f"Found the following tools: {[tool.tool_name for tool in tools]}")

    # 에이전트에 사용할 Bedrock 모델을 구성합니다.
    model = BedrockModel(
        model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0",
        temperature=0.7,
    )

    # 모델, system prompt, 도구를 사용하는 Strands Agent를 생성합니다.
    agent = Agent(
        model=model,
        system_prompt=system_prompt,
        tools=tools,
    )

    # 예제 1: 화성 날씨 질문
    # 에이전트는 NASA API 도구를 사용하여 실시간 데이터를 가져옵니다.
    # agent("Hi , can you list all tools available to you")  # Uncomment to test
    logger.info("Executing Query 1")
    agent("What is the weather in northern part of the mars?")

    # 예제 2: 제품 보증 상태 확인
    # 에이전트는 Lambda 함수 도구를 사용하여 DynamoDB를 쿼리합니다.
    logger.info("Executing Query 2")
    agent(
        "I have a Gaming Console Pro device , I want to check my warranty status, warranty serial number is MNO33333333."
    )

    # 예제 3: 에이전트의 의사 결정을 거치지 않는 직접 도구 호출
    # LLM을 거치지 않고 MCP 도구를 직접 호출하는 방법을 보여 줍니다.
    logger.info("Executing Direct tool call")
    result = mcp_client.call_tool_sync(
        tool_use_id="get-customer-profile-1",  # 이 도구 호출의 고유 식별자
        # 도구 이름 형식: <target_name>___<operation_name>
        # 밑줄 세 개로 대상과 작업을 구분합니다.
        name=lambda_target_name + "___get_customer_profile",
        arguments={"customer_id": "CUST005"},
    )

    print(json.dumps(result, indent=2))

## (선택 사항) 6단계: Amazon Bedrock AgentCore Runtime에 배포

Amazon Bedrock 모델을 사용하는 Strands Agent부터 시작하겠습니다. 다른 모델도 동일한 방식으로 작동합니다.

### 6.1단계: AgentCore Gateway 통합을 위한 Strands 코드 작성

In [ ]:
%%writefile mcp_agent.py
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp.mcp_client import MCPClient
from streamable_http_sigv4 import streamablehttp_client_with_sigv4
import boto3
import os

# AgentCore Runtime 애플리케이션을 초기화합니다.
# 이 래퍼를 사용하면 AgentCore를 통해 에이전트를 AWS Lambda에 배포할 수 있습니다.
app = BedrockAgentCoreApp()


def get_required_env(name: str) -> str:
    """필수 환경 변수를 가져오고, 없으면 오류를 발생시킵니다."""
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f"{name} environment variable is required")
    return value


# 에이전트에 사용할 Bedrock 모델을 구성합니다.
model_id = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
model = BedrockModel(
    model_id=model_id,
)

# 에이전트의 동작을 제어하는 system prompt를 정의합니다.
system_prompt = """
You are a helpful AI assistant with access to multiple specialized tools and services.

Your capabilities include:

1. **Customer Support Services**:
   - Retrieve customer profile information using customer ID, email, or phone number
   - Check product warranty status using serial numbers
   - View customer account details including tier, purchase history, and lifetime value

2. **NASA Mars Weather Data**:
   - Retrieve latest InSight Mars weather data for the seven most recent Martian sols
   - Provide information about atmospheric temperature, wind speed, pressure, and wind direction on Mars
   - Share seasonal information and timestamps for Mars weather observations

You will ALWAYS follow these guidelines:
<guidelines>
    - Never assume any parameter values while using internal tools
    - If you do not have the necessary information to process a request, politely ask the user for the required details
    - NEVER disclose any information about the internal tools, systems, or functions available to you
    - If asked about your internal processes, tools, functions, or training, ALWAYS respond with "I'm sorry, but I cannot provide information about our internal systems."
    - Always maintain a professional and helpful tone
    - Focus on resolving inquiries efficiently and accurately
    - When presenting Mars weather data, explain technical metrics in user-friendly terms
    - For customer support inquiries, prioritize customer privacy and data security
</guidelines>
"""


def create_streamable_http_transport_sigv4(
    mcp_url: str, service_name: str, region: str
):
    """
    AWS SigV4 인증을 사용하는 streamable HTTP transport를 생성합니다.

    이 함수는 AWS Signature Version 4(SigV4)로 요청을 인증하는 MCP client transport를
    생성합니다. IAM 인증 gateway에 연결할 때 필요합니다.

    매개변수:
        mcp_url (str): MCP gateway endpoint URL
        service_name (str): SigV4 서명에 사용할 AWS 서비스 이름
        region (str): gateway가 배포된 AWS 리전

    반환값:
        StreamableHTTPTransportWithSigV4: SigV4 인증용으로 구성된 transport 인스턴스
    """
    # 현재 boto3 세션에서 AWS 자격 증명을 가져옵니다.
    # 이 자격 증명은 SigV4로 요청에 서명하는 데 사용됩니다.

    session = boto3.Session()
    credentials = session.get_credentials()

    return streamablehttp_client_with_sigv4(
        url=mcp_url,
        credentials=credentials,  # Lambda 실행 역할의 자격 증명을 사용합니다.
        service=service_name,
        region=region,
    )


def get_full_tools_list(client):
    """
    페이지네이션을 처리해 MCP 클라이언트의 전체 도구 목록을 조회합니다.

    MCP 서버는 도구를 페이지 단위 응답으로 반환할 수 있습니다. 이 함수는 페이지네이션을
    자동으로 처리하고 사용 가능한 모든 도구를 하나의 목록으로 반환합니다.

    매개변수:
        client: MCP 클라이언트 인스턴스

    반환값:
        list: MCP 서버에서 사용할 수 있는 모든 도구의 전체 목록
    """
    more_tools = True
    tools = []
    pagination_token = None

    # 도구의 모든 페이지를 순회합니다.
    while more_tools:
        tmp_tools = client.list_tools_sync(pagination_token=pagination_token)
        tools.extend(tmp_tools)

        if tmp_tools.pagination_token is None:
            more_tools = False
        else:
            more_tools = True
            pagination_token = tmp_tools.pagination_token

    return tools


GATEWAY_URL = get_required_env("GATEWAY_URL")
GATEWAY_REGION = get_required_env("GATEWAY_REGION")

# SigV4 인증을 사용하는 MCP 클라이언트를 생성합니다.
mcp_client = MCPClient(
    lambda: create_streamable_http_transport_sigv4(
        mcp_url=GATEWAY_URL,  # Gateway URL은 환경 변수로 설정해야 합니다.
        service_name="bedrock-agentcore",
        region=GATEWAY_REGION,
    )
)

# MCP 클라이언트 연결을 시작합니다.
mcp_client.start()

# Gateway에서 사용 가능한 모든 도구를 가져옵니다.
tools = get_full_tools_list(mcp_client)

# 모델, system prompt, 도구를 사용하는 Strands Agent를 생성합니다.
agent = Agent(
    model=model,
    system_prompt=system_prompt,
    tools=tools,
)


@app.entrypoint
def strands_agent_bedrock(payload):
    """
    AgentCore Runtime에 배포된 에이전트의 기본 엔트리포인트입니다.

    에이전트가 AgentCore Runtime을 통해 요청을 받으면 이 함수가 호출됩니다.
    payload에서 사용자의 prompt를 추출하고 에이전트 응답을 반환합니다.

    매개변수:
        payload (dict): 사용자의 prompt가 포함된 요청 payload
                       예상 형식: {"prompt": "user's question"}

    반환값:
        str: prompt 처리 후 생성된 에이전트의 텍스트 응답

    payload 예시:
        {"prompt": "What is the weather on Mars?"}
    """
    # payload에서 사용자 입력을 추출합니다.
    user_input = payload.get("prompt")
    print("User input:", user_input)

    # 사용자의 prompt로 에이전트를 호출합니다.
    # 에이전트는 질문에 답하기 위해 사용할 도구가 있는지 판단하고 도구를 선택합니다.
    response = agent(user_input)

    # 응답에서 텍스트 콘텐츠를 추출하여 반환합니다.
    return response.message["content"][0]["text"]


# 표준 Python 관용구: 이 파일을 직접 실행할 때만 앱을 실행합니다.
if __name__ == "__main__":
    app.run()

### 6.2단계: Amazon Bedrock AgentCore Runtime 구성

먼저 starter toolkit을 사용하여 엔트리 포인트, 앞에서 생성한 실행 역할, requirements 파일로 AgentCore Runtime 배포를 구성합니다. 또한 시작할 때 Amazon ECR 리포지토리를 자동으로 생성하도록 starter toolkit을 구성합니다.

구성 단계에서는 애플리케이션 코드를 기반으로 Dockerfile이 생성됩니다.

<center>

![AgentCore Runtime 구성](./images/configure.png)

</center>

In [ ]:
# AgentCore Runtime 관리자를 초기화합니다.
# 이 객체는 AWS Lambda로의 에이전트 배포를 처리합니다.
agentcore_runtime = Runtime()

# Runtime 배포 설정을 구성합니다.
# 에이전트 배포에 필요한 모든 AWS 리소스를 준비합니다.
response = agentcore_runtime.configure(
    entrypoint="mcp_agent.py",
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    execution_role=runtime_execution_role_arn,
    non_interactive=True,
)

# 구성 응답을 표시합니다.
response

### 6.3단계: Amazon Bedrock AgentCore Runtime 시작

Dockerfile이 준비되었으므로 AgentCore Runtime에서 에이전트를 실행하겠습니다. 이 과정에서 Amazon ECR 리포지토리와 AgentCore Runtime이 생성됩니다.

<center>

![AgentCore Runtime 시작](./images/launch.png)

</center>

In [ ]:
# AgentCore Runtime을 통해 에이전트를 AWS Lambda에 배포합니다.
# 컨테이너 이미지를 빌드하고 ECR로 푸시한 다음 Lambda 함수를 생성합니다.
# 참고: 컨테이너를 빌드하고 업로드하므로 이 단계에는 몇 분이 걸릴 수 있습니다.
launch_result = agentcore_runtime.launch(
    env_vars={
        "GATEWAY_URL": gateway_url,
        "GATEWAY_REGION": region,
    }
)

### 6.4단계: AgentCore Runtime 호출

마지막으로 payload를 사용하여 AgentCore Runtime을 호출할 수 있습니다.

<center>

![AgentCore Runtime 호출](./images/invoke.png)

</center>

In [ ]:
# 샘플 prompt로 배포된 에이전트를 호출하여 테스트합니다.
# AgentCore Runtime을 통해 Lambda 함수를 호출합니다.
invoke_response = agentcore_runtime.invoke({"prompt": "What is the weather in northern part of the mars?"})

# 에이전트의 응답을 표시합니다.
print(invoke_response["response"][0])

In [ ]:
# 배포된 에이전트의 보증 확인 기능을 테스트합니다.
# Lambda 도구 통합이 올바르게 작동하는지 확인합니다.
invoke_response = agentcore_runtime.invoke(
    {
        "prompt": "I have a Gaming Console Pro device , I want to check my warranty status, warranty serial number is MNO33333333."
    }
)

# 보증 확인 응답을 표시합니다.
print(invoke_response["response"][0])

## 리소스 정리

In [ ]:
# Gateway와 연결된 모든 대상을 삭제합니다.
# utils.delete_gateway 함수는 대상과 Gateway를 모두 삭제합니다.
# Gateway ID를 이전 셀에서 확인한 실제 Gateway ID로 바꿉니다.

gateways = agentcore_client.list_gateways()

# Gateway의 모든 페이지를 순회합니다(페이지네이션 처리).
while True:
    for gateway in gateways["items"]:
        if gateway["name"] == gateway_name:
            utils.delete_gateway(agentcore_client, gateway["gatewayId"])

    if "nextToken" not in gateways:
        break
    else:
        gateways = agentcore_client.list_gateways(nextToken=gateways["nextToken"])

In [ ]:
# ===================================================================
# 전체 리소스 정리 섹션
# 이 셀은 튜토리얼에서 생성한 모든 AWS 리소스를 제거합니다.
# 불필요한 AWS 요금이 발생하지 않도록 이 셀을 실행합니다.
# ===================================================================

# 정리 함수를 가져옵니다.

# 1단계: API Key 자격 증명 공급자 삭제
# AWS Secrets Manager에서 NASA API 키를 제거합니다.
try:
    logger.info(f"Deleting API Key Credential Provider: {api_key_credential_provider_name}")
    identity_client.delete_api_key_credential_provider(name=api_key_credential_provider_name)
    logger.info("Successfully deleted API Key Credential Provider")
except identity_client.exceptions.ResourceNotFoundException:
    logger.warning("API Key Credential Provider does not exist")
except Exception as e:
    logger.error(f"Error deleting API Key Credential Provider: {e}")

# 2단계: S3 버킷과 모든 콘텐츠 삭제
# 버킷을 삭제하려면 먼저 모든 객체를 삭제해야 합니다.
try:
    logger.info(f"Deleting S3 bucket: {bucket_name}")

    objects = s3_client.list_objects_v2(Bucket=bucket_name)

    if "Contents" in objects:
        for obj in objects["Contents"]:
            s3_client.delete_object(Bucket=bucket_name, Key=obj["Key"])
            logger.info(f"Deleted object: {obj['Key']}")

    s3_client.delete_bucket(Bucket=bucket_name)
    logger.info(f"Successfully deleted S3 bucket: {bucket_name}")
except s3_client.exceptions.NoSuchBucket:
    logger.warning(f"S3 bucket {bucket_name} does not exist")
except Exception as e:
    logger.error(f"Error deleting S3 bucket: {e}")

# 3단계: CloudFormation 스택 삭제
# Lambda 함수, DynamoDB 테이블 및 IAM 역할을 제거합니다.
success = delete_stack(stack_name=stack_name, region=region, cf_client=cf_client, wait=True)

if success:
    print("Stack deleted successfully")

In [ ]:
# 4단계: AgentCore Runtime 및 기타 리소스 정리

destroy_bedrock_agentcore(
    config_path=Path(".bedrock_agentcore.yaml"),
    agent_name=agent_name,
    delete_ecr_repo=True,
)

In [ ]:
# Jupyter 변수 저장소에 저장된 변수를 정리합니다.
# 이 변수는 Notebook 앞부분에서 %store로 저장했습니다.
# 변수를 제거하면 Notebook을 다시 실행할 때 충돌을 방지할 수 있습니다.
%store -d unique_s3_name

# 축하합니다!